# MedLoRA · 实验 A: SLAKE QLoRA SFT (Kaggle T4)

流程: clone → 装依赖 + LLaMA-Factory → SLAKE 转 sharegpt → QLoRA SFT → 清理 checkpoint → 三张表评估 → 与基线对比。

训练只用 GPU 0 (单卡更稳), 预计 1–1.5 h; 评估约 1.5 h。数据集放 /kaggle/temp 不进输出。

In [ ]:
REPO_URL = "https://github.com/AugustLoo/MedLoRA.git"
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
EXP = "sft_slake_qlora_r16"

!git clone -q $REPO_URL /kaggle/working/MedLoRA
%cd /kaggle/working/MedLoRA
# 原始数据放临时盘, 不算进 notebook 输出
!mkdir -p /kaggle/temp/raw && rm -rf data/raw && ln -s /kaggle/temp/raw data/raw
!pip install -q -r requirements.txt
!pip install -q "git+https://github.com/hiyouga/LLaMA-Factory.git"
!llamafactory-cli version
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python data/download_slake.py | tail -5
!python data/convert_slake_sharegpt.py
!python -c "import json; d=json.load(open('data/processed/slake_train.json')); print('train', len(d)); print(json.dumps(d[0], ensure_ascii=False)[:400])"

In [ ]:
# 训练: 单卡, 配置见 configs/sft_slake_qlora.yaml
!CUDA_VISIBLE_DEVICES=0 llamafactory-cli train configs/sft_slake_qlora.yaml 2>&1 | grep -v -E "^\s*$|it/s\]|s/it\]" | tail -80

In [ ]:
# 训练曲线摘要 + 清理中间 checkpoint (只留最终 adapter)
import json, glob, os, shutil
log = f"outputs/{EXP}/trainer_log.jsonl"
if os.path.exists(log):
    rows = [json.loads(l) for l in open(log)]
    tr = [r for r in rows if "loss" in r]
    ev = [r for r in rows if "eval_loss" in r]
    print("train loss  first/last:", tr[0]["loss"] if tr else None, tr[-1]["loss"] if tr else None)
    print("eval  loss:", [round(r["eval_loss"], 4) for r in ev])
    bad = [r for r in tr if r["loss"] != r["loss"]]
    print("NaN steps:", len(bad))
for ck in glob.glob(f"outputs/{EXP}/checkpoint-*"):
    shutil.rmtree(ck)
!ls -la outputs/$EXP | head -20
!du -sh outputs/$EXP

In [ ]:
# 三张表评估 (加载 adapter)
!CUDA_VISIBLE_DEVICES=0 MODEL=$MODEL bash train/eval_all.sh sft_r16 outputs/$EXP 2>&1 | grep -v -E "it/s\]|s/it\]"

In [ ]:
# 与基线对比
import json
base = json.load(open("results/baseline_2026-09-15.json"))
new = {k: json.load(open(f"outputs/eval/{k}_sft_r16.json")) for k in ["slake", "textvqa", "pubmedqa"]}
def row(name, b, n): print(f"{name:<22}{b:>8}{n:>8}{n-b:>+8.2f}")
print(f"{'metric':<22}{'base':>8}{'sft_A':>8}{'delta':>8}")
for k in ["closed_acc", "open_em", "open_recall", "open_f1"]:
    row("slake_" + k, base["slake"]["metrics"][k], new["slake"]["metrics"][k])
for m in ["X-Ray", "CT", "MRI"]:
    row(f"slake_closed_{m}", base["slake"]["by_modality"][m]["closed_acc"], new["slake"]["by_modality"][m]["closed_acc"])
row("textvqa_acc", base["textvqa"]["textvqa_acc"], new["textvqa"]["textvqa_acc"])
row("pubmedqa_acc", base["pubmedqa"]["accuracy"], new["pubmedqa"]["accuracy"])
row("pubmedqa_macro_f1", base["pubmedqa"]["macro_f1"], new["pubmedqa"]["macro_f1"])
print("pubmedqa pred_dist:", new["pubmedqa"]["pred_dist"])
print("RESULTS_JSON", json.dumps(new))